# 🧑‍💻 멀티턴 대화 — 자율 실습

---

## 📌 실습 방법

- `____` 부분을 직접 채워보세요
- 막히면 💡 **힌트** 를 먼저 확인하세요
- 그래도 모르면 노트북 맨 아래 **정답 코드 셀** 을 확인하세요

```
MISSION 1. 내 도메인으로 chat() 함수 만들기
MISSION 2. 맥락 유지 확인 (3턴 대화)
MISSION 3. safe_api_call 버전으로 업그레이드
MISSION 4. 내 주제로 품질 불일치 사례 수집
MISSION 5. 히스토리 구조 분석
MISSION 6. 전체 비용 확인
```

> ⏱️ 예상 시간: **60~80분**  
> ⚠️ **MISSION 4는 Day 2에서 사용합니다. 반드시 완료하세요!**

---
## 🔧 사전 준비 — 실행만 하세요

In [ ]:
# 이전 실습과 동일한 준비 (수정 금지)
from google.colab import userdata
import openai, time, json

client = openai.OpenAI(api_key=userdata.get("OPENAI_API_KEY"))

class TokenTracker:
    def __init__(self):
        self.total_input = 0
        self.total_output = 0
        self.call_count = 0

    def add(self, response):
        self.total_input  += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count   += 1

    def report(self):
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰")
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")

tracker = TokenTracker()
print("✅ 준비 완료!")

---
## MISSION 1. 내 도메인으로 chat() 함수 만들기

강의에서는 **법률 AI** 로 만들었습니다.  
이번엔 내가 원하는 도메인(역할)으로 직접 만들어보세요.

**조건**
- `my_history` 라는 이름의 히스토리 리스트를 만드세요
- system 프롬프트는 내가 원하는 전문가 역할로 설정하세요
- `my_chat(user_input)` 함수를 4단계로 작성하세요
  1. user 메시지를 `my_history` 에 추가
  2. `my_history` 전체를 API에 전달
  3. 응답을 `my_history` 에 추가
  4. 응답 반환

> 💡 **힌트:** `list.append({"role": "user", "content": user_input})` 로 추가합니다  
> `messages=my_history` 를 API에 통째로 넘기면 됩니다

In [ ]:
# ① 히스토리 초기화 — 내가 원하는 전문가 역할로 설정
my_history = [
    {"role": "system", "content": "____"}  # 예: "당신은 파이썬 전문 강사입니다."
]

# ② my_chat 함수 — 4단계로 작성하세요
def my_chat(user_input):
    # 1단계: user 메시지를 my_history에 추가
    my_history.____({"role": "____", "content": user_input})

    # 2단계: my_history 전체를 API에 전달
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=____,   # 히스토리 전체를 넘기세요
        temperature=0.3,
        max_tokens=300,
    )

    # 3단계: 응답을 my_history에 추가
    reply = response.choices[0].message.content
    my_history.____({"role": "____", "content": reply})

    # 4단계: 비용 추적 + 응답 반환
    tracker.add(response)
    return ____

print("✅ MISSION 1 완료!")

---
## MISSION 2. 맥락 유지 확인 (3턴 대화)

MISSION 1에서 만든 `my_chat()` 으로 **3턴 대화**를 실행하세요.  
**핵심:** 3번째 질문은 반드시 "방금 말한 내용" 이나 "앞에서 언급한" 처럼 이전 대화를 참조해야 합니다.

**조건**
- 질문 1: 내 도메인의 일반적인 질문
- 질문 2: 질문 1의 답변을 이어가는 질문 ("그렇다면...", "그 중에서...")
- 질문 3: "방금 말한 내용을 요약해 주세요" 또는 "앞에서 말한 내용 정리해줘"

> 💡 **힌트:** 3번째 응답에 1·2턴 내용이 담겨야 맥락 연결 성공입니다

In [ ]:
# 3턴 대화 실행
print("=" * 50)
print("[1턴]", my_chat("____"))   # 첫 번째 질문

print("\n" + "=" * 50)
print("[2턴]", my_chat("____"))   # 이어지는 질문 (그렇다면... 등)

print("\n" + "=" * 50)
print("[3턴]", my_chat("방금 말한 내용을 요약해 주세요."))  # 맥락 참조 확인

In [ ]:
# 히스토리 구조 확인
print(f"히스토리 길이: {len(my_history)}개")   # 7이어야 합니다 (system + 3턴 × 2)

roles = [item["role"] for item in my_history]   # role 목록 추출
print(f"role 순서: {roles}")   # system, user, assistant 가 반복되어야 합니다

---
## MISSION 3. safe_api_call 버전으로 업그레이드

MISSION 1의 `my_chat()` 을 **API 실패에도 안전한 버전**으로 업그레이드하세요.

**조건**
- `safe_api_call()` 함수를 먼저 만들고
- `my_chat_safe()` 함수에서 이를 활용하세요
- API 실패 시 `my_history.pop()` 으로 히스토리를 원래대로 복구하세요

> 💡 **힌트:**  
> `safe_api_call(messages)` 는 성공 시 응답 텍스트, 실패 시 `None` 을 반환합니다  
> `reply is None` 이면 `my_history.pop()` 후 에러 메시지를 반환하세요

In [ ]:
# ① 안전한 호출 함수
def safe_api_call(messages, temperature=0.3, max_tokens=300):
    for attempt in range(3):   # 최대 3번 시도
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            tracker.add(response)
            return ____   # 성공 시 응답 텍스트 반환

        except Exception as e:
            wait = 2 ** attempt   # 1초 → 2초 → 4초
            print(f"⚠️ [{attempt+1}/3] {e} → {wait}초 후 재시도")
            time.sleep(wait)

    return ____   # 3번 모두 실패하면 None 반환


# ② safe 버전 chat 함수
def my_chat_safe(user_input):
    my_history.append({"role": "user", "content": user_input})   # user 추가

    reply = safe_api_call(____)   # my_history 전달

    if reply is ____:   # 실패 시
        my_history.____()   # 방금 추가한 user 메시지 제거
        return "⚠️ 응답을 받지 못했습니다. 다시 시도해 주세요."

    my_history.append({"role": "assistant", "content": reply})
    return reply


# 테스트
my_history = [{"role": "system", "content": "당신은 파이썬 전문 강사입니다."}]   # 초기화
print(my_chat_safe("파이썬 리스트와 튜플의 차이는 무엇인가요?"))

---
## MISSION 4. 내 주제로 품질 불일치 사례 수집

> ⚠️ **이 미션은 Day 2에서 사용합니다. 반드시 완료하고 저장하세요!**

내가 관심 있는 도메인의 질문으로 **같은 설정, 3회 호출**을 해보세요.  
답변이 매번 같은지, 다른지 관찰하고 기록합니다.

**조건**
- 질문은 아래 예시 중 선택하거나 자유롭게 정하세요
- `temperature=0.3`, 동일 조건으로 **3회 호출**
- 3개 응답의 차이점을 분석하세요
- Drive에 JSON으로 저장하세요

| # | 질문 예시 |
|---|----------|
| A | "근로계약서 없이 일한 경우 임금 체불 시 어떻게 대처해야 하나요?" |
| B | "파이썬을 독학으로 배우는 가장 효과적인 방법은?" |
| C | 직접 작성 |

> 💡 **힌트:** `quality_samples = []` 리스트에 3회 응답을 저장하고  
> `for i in range(3):` 로 반복 호출합니다

In [ ]:
# 내가 선택한 질문
my_question = "____"   # 위 예시 중 선택 또는 직접 작성

my_samples = []   # 3개 응답을 저장할 리스트

for i in range(____):   # 3번 반복
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 전문 AI입니다. 정확하고 친절하게 답변하세요."},
            {"role": "user",   "content": ____},   # my_question 넣기
        ],
        temperature=____,   # 0.3으로 설정
        max_tokens=300,
    )
    answer = ____   # 응답 텍스트 꺼내기
    my_samples.append(answer)   # 리스트에 저장
    tracker.add(response)

    print(f"[{i+1}회차]")
    print(answer[:200])
    print()

In [ ]:
# 3개 응답의 차이를 분석하세요
print("📋 품질 불일치 분석")
print("=" * 50)

for i, ans in enumerate(my_samples):
    has_structure = any(c in ans for c in ["1.", "①", "첫째", "- "])   # 구조화 여부

    print(f"[{i+1}회차] 길이: {____}자  구조화: {'✅' if has_structure else '❌'}")   # len() 사용
    print(f"  시작: {ans[:60]}...")

# 동일 여부
if my_samples[0] == my_samples[1] == my_samples[2]:
    print("\n→ 3회 모두 동일")
else:
    print("\n→ ⚠️ 답변이 다릅니다! 이것이 품질 불일치입니다.")

In [ ]:
# Drive에 저장 (Day 2에서 사용)
from google.colab import drive
drive.mount("/content/drive")

import os
os.makedirs("/content/drive/MyDrive/llm_course/day1", exist_ok=True)

quality_note = {
    "question": my_question,
    "samples": my_samples,
    "observation": "____"   # 관찰한 내용을 직접 작성하세요
}

with open("/content/drive/MyDrive/llm_course/day1/quality_note.json", "w", encoding="utf-8") as f:
    json.dump(quality_note, f, ensure_ascii=False, indent=2)

print("✅ 저장 완료 → Day 2에서 이 파일을 사용합니다!")

**📝 품질 불일치 기록표 — 직접 채우세요**

| 항목 | 1회차 | 2회차 | 3회차 |
|------|-------|-------|-------|
| 답변 구조 (목록형/서술형) | | | |
| 답변 길이 (글자 수) | | | |
| 정확도 (상/중/하) | | | |
| 핵심 차이점 | | | |

---
## MISSION 5. 히스토리 구조 분석

MISSION 2에서 3턴 대화를 했을 때 히스토리가 어떻게 쌓였는지 직접 확인해보세요.

**출력해야 할 것 (3가지)**
1. 히스토리 총 길이 (`len()`)
2. 각 메시지의 role 목록 (리스트 컴프리헨션)
3. 마지막 응답 내용 앞 100자 (`[-1]` 로 마지막 항목 접근)

> 💡 **힌트:** `[item["role"] for item in my_history]` 로 role 목록을 만들 수 있습니다

In [ ]:
# ① 히스토리 총 길이
print(f"히스토리 길이: {____}개")

# ② role 목록
roles = [item[____] for item in my_history]   # "role" 키를 꺼내세요
print(f"role 순서: {roles}")

# ③ 마지막 응답 앞 100자
print(f"\n마지막 응답:")
print(my_history[____]["content"][:100])   # -1 로 마지막 항목 접근

---
## MISSION 6. 전체 비용 확인

MISSION 1~5에서 사용한 **전체 누적 비용**을 출력하세요.

> 💡 **힌트:** `tracker.report()` 한 줄이면 됩니다

In [ ]:
# 전체 누적 비용 출력
print("💰 전체 실습 누적 비용")
____

---
## 🌟 도전 과제

### 도전 1. 히스토리 초기화 함수 만들기
- `reset_my_chat()` 함수를 만드세요
- 호출하면 `my_history` 가 system만 남은 초기 상태로 돌아와야 합니다
- 초기화 후 새 대화를 3턴 시작해보세요

### 도전 2. 턴별 토큰 증가량 추적
- `my_chat_safe()` 를 수정해서 매 턴마다 입력 토큰 수를 출력하세요
- `response.usage.prompt_tokens` 를 활용하세요
- 5턴 대화 후 입력 토큰이 어떻게 증가하는지 확인하세요

### 도전 3. 트리밍 함수 구현 (심화)
- `chat_with_trim(user_input, max_turns=3)` 함수를 만드세요
- system은 항상 유지하되, 최근 `max_turns` 턴만 보존하세요
- **힌트:** `history = [history[0]] + history[-(max_turns * 2):]`

In [ ]:
# 도전 과제를 여기에 자유롭게 작성하세요


---
## 📂 정답 코드

> ⚠️ 꼭 혼자 먼저 시도한 뒤에 확인하세요!

In [ ]:
# ✅ MISSION 1 정답 ─ chat() 함수 만들기
my_history = [
    {"role": "system", "content": "당신은 파이썬 전문 강사입니다. 쉽게 설명하세요."}
]

def my_chat(user_input):
    my_history.append({"role": "user", "content": user_input})       # 1단계: user 추가

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=my_history,     # 2단계: 히스토리 전체 전달
        temperature=0.3,
        max_tokens=300,
    )

    reply = response.choices[0].message.content
    my_history.append({"role": "assistant", "content": reply})        # 3단계: assistant 추가

    tracker.add(response)
    return reply                                                       # 4단계: 반환

print("✅ my_chat() 준비 완료")

In [ ]:
# ✅ MISSION 2 정답 ─ 3턴 대화 + 히스토리 확인
print("=" * 50)
print("[1턴]", my_chat("파이썬 리스트란 무엇인가요?"))

print("\n" + "=" * 50)
print("[2턴]", my_chat("그렇다면 딕셔너리는 리스트와 어떻게 다른가요?"))

print("\n" + "=" * 50)
print("[3턴]", my_chat("방금 말한 내용을 요약해 주세요."))   # 맥락 참조 확인

# 히스토리 구조 확인
print(f"\n히스토리 길이: {len(my_history)}개")
roles = [item["role"] for item in my_history]
print(f"role 순서: {roles}")

In [ ]:
# ✅ MISSION 3 정답 ─ safe 버전
def safe_api_call(messages, temperature=0.3, max_tokens=300):
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            tracker.add(response)
            return response.choices[0].message.content   # 성공 시 반환
        except Exception as e:
            wait = 2 ** attempt
            print(f"⚠️ [{attempt+1}/3] {e} → {wait}초 후 재시도")
            time.sleep(wait)
    return None   # 실패 시 None


def my_chat_safe(user_input):
    my_history.append({"role": "user", "content": user_input})

    reply = safe_api_call(my_history)   # my_history 전달

    if reply is None:             # 실패 시
        my_history.pop()          # user 메시지 제거 (히스토리 보호)
        return "⚠️ 응답을 받지 못했습니다."

    my_history.append({"role": "assistant", "content": reply})
    return reply


# 테스트
my_history = [{"role": "system", "content": "당신은 파이썬 전문 강사입니다."}]
print(my_chat_safe("파이썬 리스트와 튜플의 차이는?"))

In [ ]:
# ✅ MISSION 4 정답 ─ 품질 불일치 수집
my_question = "파이썬을 독학으로 배우는 가장 효과적인 방법은?"

my_samples = []

for i in range(3):   # 3번 반복
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "당신은 전문 AI입니다."},
            {"role": "user",   "content": my_question},
        ],
        temperature=0.3,
        max_tokens=300,
    )
    answer = response.choices[0].message.content   # 응답 텍스트
    my_samples.append(answer)
    tracker.add(response)
    print(f"[{i+1}회차]\n{answer[:200]}\n")

# 분석
for i, ans in enumerate(my_samples):
    has_structure = any(c in ans for c in ["1.", "①", "첫째", "- "])
    print(f"[{i+1}회차] 길이: {len(ans)}자  구조화: {'✅' if has_structure else '❌'}")

# Drive 저장
quality_note = {"question": my_question, "samples": my_samples, "observation": "관찰 내용"}
with open("/content/drive/MyDrive/llm_course/day1/quality_note.json", "w", encoding="utf-8") as f:
    json.dump(quality_note, f, ensure_ascii=False, indent=2)
print("✅ 저장 완료")

In [ ]:
# ✅ MISSION 5 정답 ─ 히스토리 구조 분석
print(f"히스토리 길이: {len(my_history)}개")

roles = [item["role"] for item in my_history]   # role 목록
print(f"role 순서: {roles}")

print(f"\n마지막 응답:")
print(my_history[-1]["content"][:100])   # -1 로 마지막 항목

In [ ]:
# ✅ MISSION 6 정답 ─ 전체 비용
print("💰 전체 실습 누적 비용")
tracker.report()   # 한 줄